# UNSW-NB15 — Task 2 Preprocessing (GNN-ready train/test split)
Group 5 · CSE475 · Track 1 (GNN)

This notebook takes the raw, deduplicated UNSW-NB15 flow data and prepares leakage-safe
train/test splits ready for GNN model input. Every *fitted* transform (fill values,
encoders, scaler, class weights) is fit on **train only** and applied to test — that's
the single rule that matters most in this notebook, and it comes up at nearly every step.

**Pipeline order:** load raw + schema → global dedup → clean `attack_cat` → sequester
IP/port/time columns into a side table → host- or time-based split (no shuffling) →
per-split cleaning (missing values → dedup check → encode → scale → drop flagged
columns → drop IP/port/time columns) → handle imbalance (train only) → save outputs →
final summary.

## 0. Setup
Imports + locate the data folder. Edit `DATA_DIR` below if auto-detect doesn't find your
files — e.g. after mounting Google Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/<your folder>'
```

In [4]:
import os, glob, pickle, warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

RANDOM_SEED = 42

In [5]:
# EDIT THIS if auto-detect below doesn't find your files - point it straight at the
# folder containing NUSW-NB15_features.csv + the 4 raw CSVs.
DATA_DIR = '/content'

def find_data_dir(start_dirs, filename='NUSW-NB15_features.csv'):
    """Search a few likely locations for the feature-schema file and return its folder."""
    for start in start_dirs:
        if not os.path.isdir(start):
            continue
        hits = glob.glob(os.path.join(start, '**', filename), recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    return None

# Covers plain Colab upload (/content), Drive-mounted Colab (both possible Drive mount
# spellings), the current working directory, and Kaggle - in case this gets run there too.
SEARCH_DIRS = [
    DATA_DIR, '/content', '/content/drive/MyDrive', '/content/drive/My Drive',
    os.getcwd(), '/kaggle/input',
]
found = find_data_dir(SEARCH_DIRS)

if found:
    DATA_DIR = found
    print("Using DATA_DIR:", DATA_DIR)
    print(os.listdir(DATA_DIR))
else:
    # Fail loudly HERE with a clear diagnostic, instead of silently falling through to a
    # confusing FileNotFoundError two cells later.
    print("Could not find NUSW-NB15_features.csv in any of:")
    for d in SEARCH_DIRS:
        print("  -", d, "(exists)" if os.path.isdir(d) else "(does not exist)")
    if os.path.isdir('/content'):
        print("\nWhat's actually in /content:", os.listdir('/content'))
    raise FileNotFoundError(
        "Upload the 4 CSVs + NUSW-NB15_features.csv to this Colab session (the folder icon "
        "on the left sidebar), or mount Drive and set DATA_DIR to the exact folder above, "
        "then re-run this cell."
    )


Using DATA_DIR: /kaggle/input/datasets/shahiismyname/unsw-nb15
['UNSW-NB15_1.csv', 'UNSW-NB15_4.csv', 'UNSW-NB15_3.csv', 'UNSW-NB15_2.csv', 'NUSW-NB15_features.csv']


## 1. Load feature schema, load raw flow files with proper headers
`UNSW-NB15_1.csv`–`_4.csv` have **no header row**. `NUSW-NB15_features.csv` is the only
place that defines the exact column names and their order, so it has to be loaded first
— its `Name` column becomes the `names=` argument when reading the four raw files.

In [6]:
def load_feature_schema(schema_path):
    """Load NUSW-NB15_features.csv and return an ordered list of column names."""
    feat = pd.read_csv(schema_path, encoding='latin1')
    feat.columns = [c.strip() for c in feat.columns]
    feat['Name'] = feat['Name'].astype(str).str.strip().str.replace(' ', '', regex=False)
    feat['Type'] = feat['Type'].astype(str).str.strip().str.lower()
    return feat['Name'].tolist(), feat

def load_raw_flows(raw_files, col_names):
    """Load and concatenate the raw (headerless) flow CSVs using the schema's column names."""
    dfs = [pd.read_csv(f, header=None, names=col_names, low_memory=False) for f in raw_files]
    return pd.concat(dfs, ignore_index=True)

col_names, feat_schema = load_feature_schema(os.path.join(DATA_DIR, 'NUSW-NB15_features.csv'))
print(f"{len(col_names)} columns expected:", col_names)

raw_files = sorted(glob.glob(os.path.join(DATA_DIR, 'UNSW-NB15_[1-4].csv')))
print("Raw files found:", raw_files)
assert len(raw_files) == 4, f"Expected 4 raw files, found {len(raw_files)} - check DATA_DIR"

df = load_raw_flows(raw_files, col_names)
print("Combined shape (pre-dedup):", df.shape)
df.head()

49 columns expected: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']
Raw files found: ['/kaggle/input/datasets/shahiismyname/unsw-nb15/UNSW-NB15_1.csv', '/kaggle/input/datasets/shahiismyname/unsw-nb15/UNSW-NB15_2.csv', '/kaggle/input/datasets/shahiismyname/unsw-nb15/UNSW-NB15_3.csv', '/kaggle/input/datasets/shahiismyname/unsw-nb15/UNSW-NB15_4.csv']
Combined shape (pre-dedup): (2540047, 49)


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,service,Sload,Dload,Spkts,Dpkts,swin,dwin,stcpb,dtcpb,smeansz,dmeansz,trans_depth,res_bdy_len,Sjit,Djit,Stime,Ltime,Sintpkt,Dintpkt,tcprtt,synack,ackdat,is_sm_ips_ports,ct_state_ttl,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,29,0,0,dns,500473.93750,621800.93750,2,2,0,0,0,0,66,82,0,0,0.00000,0.000000,1421927414,1421927414,0.017,0.013000,0.0,0.0,0.0,0,0,0.0,0.0,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,29,0,0,-,87676.08594,50480.17188,4,4,0,0,0,0,132,76,0,0,9.89101,10.682733,1421927414,1421927414,7.005,7.564333,0.0,0.0,0.0,0,0,0.0,0.0,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,29,0,0,dns,521894.53130,636282.37500,2,2,0,0,0,0,73,89,0,0,0.00000,0.000000,1421927414,1421927414,0.017,0.013000,0.0,0.0,0.0,0,0,0.0,0.0,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,29,0,0,dns,436724.56250,542597.18750,2,2,0,0,0,0,66,82,0,0,0.00000,0.000000,1421927414,1421927414,0.043,0.014000,0.0,0.0,0.0,0,0,0.0,0.0,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,29,0,0,dns,499572.25000,609067.56250,2,2,0,0,0,0,73,89,0,0,0.00000,0.000000,1421927414,1421927414,0.005,0.003000,0.0,0.0,0.0,0,0,0.0,0.0,0,7,9,1,1,1,1,1,NaN,0


## 2. Deduplicate across all four files — BEFORE any split
Task 1 EDA found **18.92% exact duplicate rows** in the raw data. Deduplicating once,
globally, before the split keeps every downstream count honest — duplicates don't get a
chance to inflate one class over another or land on both sides of a host/time boundary
in a way that's hard to reason about later.

Right after dedup we assign a permanent `flow_id`. Every later step (split, side table,
saved files) refers back to this ID, so nothing gets silently misaligned between
teammates' copies of the data.

In [7]:
def dedup_global(df):
    before = len(df)
    df = df.drop_duplicates(ignore_index=True)
    after = len(df)
    print(f"Dedup: {before:,} -> {after:,} rows  ({before - after:,} removed, "
          f"{100 * (before - after) / before:.2f}%)")
    return df, before, after

df, rows_before_dedup, rows_after_dedup = dedup_global(df)

df = df.reset_index(drop=True)
df['flow_id'] = df.index

Dedup: 2,540,047 -> 2,059,415 rows  (480,632 removed, 18.92%)


## 3. Clean `attack_cat`
Blank/NaN entries mean normal traffic → fill with `"Normal"`. The raw files also mix
casing/spacing for the same category (e.g. `" Fuzzers"` vs `"Fuzzers"`, `"Backdoor"` vs
`"Backdoors"`) — normalize those before anything downstream counts classes.

In [8]:
def clean_attack_cat(df, casing_fixes=None):
    df = df.copy()
    df['attack_cat'] = df['attack_cat'].astype(str).str.strip()
    df['attack_cat'] = df['attack_cat'].replace({'nan': 'Normal', '': 'Normal'})
    df['attack_cat'] = df['attack_cat'].str.replace(r'\s+', ' ', regex=True)
    if casing_fixes:
        df['attack_cat'] = df['attack_cat'].replace(casing_fixes)
    return df

# Known casing/naming variants from Task 1 EDA - extend if step 4 below turns up more.
CASING_FIXES = {'Backdoor': 'Backdoors'}

df = clean_attack_cat(df, CASING_FIXES)
df['Label'] = pd.to_numeric(df['Label'], errors='coerce')
print(sorted(df['attack_cat'].unique()))

['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


## 4. Confirm rows, columns, dtypes, class balance
Sanity check before anything else happens to the data.

In [9]:
def summarize_dataset(df, label='dataset'):
    print(f"--- {label} ---")
    print("Shape:", df.shape)
    print("\nDtype counts:\n", df.dtypes.value_counts())
    print("\nattack_cat counts:\n", df['attack_cat'].value_counts())
    print("\nLabel counts:\n", df['Label'].value_counts())

summarize_dataset(df, "full dataset (post-dedup, pre-split)")

--- full dataset (post-dedup, pre-split) ---
Shape: (2059415, 50)

Dtype counts:
 int64      29
float64    12
object      9
Name: count, dtype: int64

attack_cat counts:
 attack_cat
Normal            1959772
Exploits            27599
Generic             25378
Fuzzers             21795
Reconnaissance      13357
DoS                  5665
Analysis             2184
Backdoors            1983
Shellcode            1511
Worms                 171
Name: count, dtype: int64

Label counts:
 Label
0    1959772
1      99643
Name: count, dtype: int64


## 5. Sequester IP/port/time columns into a side table
`srcip`, `sport`, `dstip`, `dsport`, `Stime`, `Ltime` are **never model features** — raw
IPs/ports would let the model memorize specific hosts instead of learning generalizable
traffic patterns, and they won't even exist for a host the model hasn't seen before. But
they're not dropped yet: `Stime`/`srcip`/`dstip` are needed for the split logic in step 6,
and all six are needed later for graph construction. So they get copied into a side table
now (linked back by `flow_id`), while the main `df` keeps them a little longer — they're
only actually dropped from the model-input frames in step 7g.

One dataset quirk worth handling here: some `sport`/`dsport` values in the raw UNSW-NB15
files are hex strings (e.g. `'0x000d'`) instead of decimal — normalized below so the side
table's ports are usable for grouping/graph-edge construction later.

In [10]:
ID_TIME_COLS = ['srcip', 'sport', 'dstip', 'dsport', 'Stime', 'Ltime']

def clean_port(x):
    """Normalize a sport/dsport value (decimal or hex string) to a plain int."""
    try:
        s = str(x).strip()
        return int(s, 16) if s.lower().startswith('0x') else int(float(s))
    except (ValueError, TypeError):
        return np.nan

def build_side_table(df, id_time_cols=ID_TIME_COLS):
    side = df[['flow_id'] + id_time_cols].copy()
    side['sport'] = side['sport'].apply(clean_port)
    side['dsport'] = side['dsport'].apply(clean_port)
    return side

side_table = build_side_table(df)
print("Side table shape:", side_table.shape)
side_table.head()

Side table shape: (2059415, 7)


,flow_id,srcip,sport,dstip,dsport,Stime,Ltime
0,0,59.166.0.0,1390.0,149.171.126.6,53.0,1421927414,1421927414
1,1,59.166.0.0,33661.0,149.171.126.9,1024.0,1421927414,1421927414
2,2,59.166.0.6,1464.0,149.171.126.7,53.0,1421927414,1421927414
3,3,59.166.0.5,3593.0,149.171.126.5,53.0,1421927414,1421927414
4,4,59.166.0.3,49664.0,149.171.126.0,53.0,1421927414,1421927414


## 6. Train/test split — host-based or time-based, no shuffling
No random shuffling, because network flows are correlated in time and by host. A random
row-level split would let flows from the same host, or the same short time window, land
on both sides — the model would partly be tested on data it effectively already saw,
producing an overoptimistic evaluation. Splitting by whole host or by time window closes
that leak, and time-based split additionally mirrors how the model would really be used
(train on past traffic, evaluate on traffic that comes after).

**Caveat from Task 1 EDA:** the whole dataset only has **49 unique hosts**. A host-based
split with that few hosts can easily strand a rare class (`Worms` = 174 rows total)
almost entirely on one side, or leave one side with very few hosts. Time-based split
doesn't have that failure mode, so it's the default below — `split_host_based` is
provided too, but check the printed class balance carefully before switching to it.

In [11]:
def split_time_based(df, test_frac=0.2, time_col='Stime'):
    """Earlier time_col window = train, later = test. No shuffling."""
    df_sorted = df.sort_values(time_col, kind='mergesort').reset_index(drop=True)
    cutoff = int(len(df_sorted) * (1 - test_frac))
    train_ids = set(df_sorted.iloc[:cutoff]['flow_id'])
    test_ids = set(df_sorted.iloc[cutoff:]['flow_id'])
    return train_ids, test_ids

def split_host_based(df, test_frac=0.2, seed=RANDOM_SEED):
    """Every flow touching a given host (as src OR dst) goes entirely to one side."""
    hosts = pd.unique(pd.concat([df['srcip'], df['dstip']]))
    rng = np.random.default_rng(seed)
    hosts = rng.permutation(hosts)
    n_test_hosts = max(1, int(len(hosts) * test_frac))
    test_hosts = set(hosts[:n_test_hosts])
    is_test = df['srcip'].isin(test_hosts) | df['dstip'].isin(test_hosts)
    train_ids = set(df.loc[~is_test, 'flow_id'])
    test_ids = set(df.loc[is_test, 'flow_id'])
    return train_ids, test_ids

# 'time' or 'host' - see markdown above for why 'time' is the default given only 49 hosts.
SPLIT_STRATEGY = 'time'

if SPLIT_STRATEGY == 'time':
    train_ids, test_ids = split_time_based(df, test_frac=0.2)
else:
    train_ids, test_ids = split_host_based(df, test_frac=0.2)

train_df = df[df['flow_id'].isin(train_ids)].reset_index(drop=True)
test_df = df[df['flow_id'].isin(test_ids)].reset_index(drop=True)

print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Train: {len(train_df):,} rows   Test: {len(test_df):,} rows")
print("\nTrain attack_cat balance:\n", train_df['attack_cat'].value_counts(normalize=True).round(4))
print("\nTest attack_cat balance:\n", test_df['attack_cat'].value_counts(normalize=True).round(4))

Split strategy: time
Train: 1,647,532 rows   Test: 411,883 rows

Train attack_cat balance:
 attack_cat
Normal            0.9598
Exploits          0.0113
Generic           0.0098
Fuzzers           0.0090
Reconnaissance    0.0054
DoS               0.0023
Analysis          0.0009
Backdoors         0.0008
Shellcode         0.0006
Worms             0.0001
Name: proportion, dtype: float64

Test attack_cat balance:
 attack_cat
Normal            0.9189
Generic           0.0226
Exploits          0.0219
Fuzzers           0.0167
Reconnaissance    0.0108
DoS               0.0044
Analysis          0.0017
Backdoors         0.0017
Shellcode         0.0012
Worms             0.0001
Name: proportion, dtype: float64


## 7. Per-split cleaning
Everything from here on runs **separately on `train_df` and `test_df`**. Any step that
*fits* a value (a fill value, an encoder, a scaler) fits on train only and reuses that
same fitted value on test — test never gets to influence how it's transformed.

### 7a. Missing values

In [12]:
def missing_report(df, label='dataset'):
    miss = (df.isna().mean() * 100).sort_values(ascending=False)
    miss = miss[miss > 0]
    print(f"--- Missing values: {label} ---")
    print(miss if len(miss) else "(none)")
    return miss

train_missing_before = missing_report(train_df, "train (pre-fill)")
test_missing_before = missing_report(test_df, "test (pre-fill)")

--- Missing values: train (pre-fill) ---
is_ftp_login        37.084196
ct_flw_http_mthd    34.393262
dtype: float64
--- Missing values: test (pre-fill) ---
is_ftp_login        97.889692
ct_flw_http_mthd    89.093990
dtype: float64


### 7b. Fill missing values — fit on train only
Fitting fill values on train only isn't just procedure — computing a median or a "most
common category" from the combined train+test data would leak test statistics into
training, which is the same kind of leak a random split causes, just one step later.

**One deviation from a blanket "median for numeric" rule, based on Task 1 EDA:** the only
two columns with any missing values are `is_ftp_login` (56%) and `ct_flw_http_mthd` (53%),
and in both cases NaN means *"this flow isn't ftp/http traffic"* — that's structural, not
random missingness. Filling those with the train median would invent a fake typical
ftp/http value for flows that never touched ftp/http at all, so they get filled with `0`
instead; every other numeric column still uses the train median.

In [13]:
def compute_fill_values(train_df, numeric_cols, categorical_cols, structural_zero_cols=None):
    structural_zero_cols = structural_zero_cols or []
    fill_values = {}
    for c in numeric_cols:
        fill_values[c] = 0 if c in structural_zero_cols else train_df[c].median()
    for c in categorical_cols:
        fill_values[c] = 'unknown'
    return fill_values

def apply_fill(df, fill_values):
    return df.fillna(fill_values)

# Structural NaN, not random missingness - see markdown above.
STRUCTURAL_ZERO_COLS = ['is_ftp_login', 'ct_flw_http_mthd']

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ID_TIME_COLS + ['flow_id', 'Label']]

categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ID_TIME_COLS + ['attack_cat']]

fill_values = compute_fill_values(train_df, numeric_cols, categorical_cols, STRUCTURAL_ZERO_COLS)

train_df = apply_fill(train_df, fill_values)
test_df = apply_fill(test_df, fill_values)  # same values, computed from train only

_ = missing_report(train_df, "train (post-fill)")
_ = missing_report(test_df, "test (post-fill)")

--- Missing values: train (post-fill) ---
(none)
--- Missing values: test (post-fill) ---
(none)


### 7c. Deduplicate train and test separately
Belt-and-suspenders: the global dedup in step 2 already removed every exact duplicate
before the split, and splitting or filling can't create new ones — so this should find
nothing. It's kept as an explicit, printed check rather than assumed silently.

In [14]:
def dedup_check(df, label):
    dup = df.duplicated().sum()
    if dup:
        print(f"{label}: removing {dup} residual duplicate rows")
        df = df.drop_duplicates(ignore_index=True)
    else:
        print(f"{label}: no residual duplicates (expected - global dedup already ran in step 2)")
    return df

train_df = dedup_check(train_df, "train")
test_df = dedup_check(test_df, "test")

train: no residual duplicates (expected - global dedup already ran in step 2)
test: no residual duplicates (expected - global dedup already ran in step 2)


### 7d. Encode categorical columns — fit on train only
`proto`, `service`, `state` get ordinal-encoded. The encoder is fit on train only; test
is transformed with that same fitted encoder. `handle_unknown='use_encoded_value'` maps
any category seen in test but never seen in train to `-1` instead of crashing — a
category genuinely never occurring in train has no fitted representation to give it, but
the flow shouldn't be dropped just because of that.

In [15]:
CATEGORICAL_FEATURE_COLS = ['proto', 'service', 'state']  # edit if your schema differs

def fit_encoder(train_df, cat_cols):
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    enc.fit(train_df[cat_cols].astype(str))
    return enc

def apply_encoder(df, enc, cat_cols):
    df = df.copy()
    df[cat_cols] = enc.transform(df[cat_cols].astype(str))
    return df

cat_encoder = fit_encoder(train_df, CATEGORICAL_FEATURE_COLS)
train_df = apply_encoder(train_df, cat_encoder, CATEGORICAL_FEATURE_COLS)
test_df = apply_encoder(test_df, cat_encoder, CATEGORICAL_FEATURE_COLS)

print("Encoded categories per column:")
for col, cats in zip(CATEGORICAL_FEATURE_COLS, cat_encoder.categories_):
    print(f"  {col}: {len(cats)} categories")

Encoded categories per column:
  proto: 135 categories
  service: 13 categories
  state: 16 categories


### 7e. Scale numeric columns — fit on train only
Same reasoning as the encoder: `StandardScaler` fit on train's mean/std only, then
applied to both. Fitting on the combined data would leak test's scale into training.

In [16]:
def fit_scaler(train_df, num_cols):
    scaler = StandardScaler()
    scaler.fit(train_df[num_cols])
    return scaler

def apply_scaler(df, scaler, num_cols):
    df = df.copy()
    df[num_cols] = scaler.transform(df[num_cols])
    return df

scaler = fit_scaler(train_df, numeric_cols)
train_df = apply_scaler(train_df, scaler, numeric_cols)
test_df = apply_scaler(test_df, scaler, numeric_cols)

print("Scaled", len(numeric_cols), "numeric columns.")

Scaled 37 numeric columns.


### 7f. Drop near-constant and highly-correlated columns
The list below is filled in directly from the Task 1 EDA numbers — edit it if your
report's final list differs.

- **Near-constant:** `is_sm_ips_ports` was 99.83% a single value.
- **High-correlation pairs (|r| > 0.9):** for each pair, one column is dropped and the
  other kept. `Stime`/`Ltime` (r = 1.0) isn't repeated here since both are already
  sequestered in the side table. `sttl`/`Label` (r = 0.90) is **deliberately not** treated
  as redundancy — a feature correlating with the *target* is a useful signal, not
  duplication, so `sttl` is kept.

In [17]:
NEAR_CONSTANT_COLS = ['is_sm_ips_ports']

# Each line = which column is dropped, and which correlated partner(s) from Task 1 EDA it resolves.
HIGH_CORR_DROP_COLS = [
    'dwin',               # swin/dwin r=.997 -> keep swin
    'Dpkts',              # dloss/Dpkts r=.992, dbytes/Dpkts r=.971 -> keep dbytes, dloss(see below)
    'dloss',              # dbytes/dloss r=.991 -> keep dbytes
    'sloss',              # sbytes/sloss r=.953 -> keep sbytes
    'ct_src_dport_ltm',   # hub: correlated .91-.96 with ct_dst_ltm, ct_src_ltm, ct_dst_sport_ltm, ct_dst_src_ltm
    'ct_dst_ltm',         # ct_dst_ltm/ct_src_ltm r=.939 -> keep ct_src_ltm
    'ct_srv_dst',         # ct_srv_src/ct_srv_dst r=.957 -> keep ct_srv_src
    'ct_dst_src_ltm',     # ct_srv_src/ct_dst_src_ltm r=.942, ct_srv_dst/ct_dst_src_ltm r=.951
    'synack',             # tcprtt/synack r=.931 -> keep tcprtt
    'ackdat',             # tcprtt/ackdat r=.919 -> keep tcprtt
    'ct_state_ttl',       # sttl/ct_state_ttl r=.906 -> keep sttl (also predictive of Label)
]

DROP_COLS = [c for c in NEAR_CONSTANT_COLS + HIGH_CORR_DROP_COLS if c in train_df.columns]

def drop_flagged_columns(df, drop_cols):
    return df.drop(columns=drop_cols, errors='ignore')

train_df = drop_flagged_columns(train_df, DROP_COLS)
test_df = drop_flagged_columns(test_df, DROP_COLS)

print("Dropped:", DROP_COLS)
print("Train shape now:", train_df.shape, "  Test shape now:", test_df.shape)

Dropped: ['is_sm_ips_ports', 'dwin', 'Dpkts', 'dloss', 'sloss', 'ct_src_dport_ltm', 'ct_dst_ltm', 'ct_srv_dst', 'ct_dst_src_ltm', 'synack', 'ackdat', 'ct_state_ttl']
Train shape now: (1647532, 38)   Test shape now: (411883, 38)


### 7g. Drop IP/port/time columns from the model-input frames
The split logic in step 6 needed `srcip`/`dstip`/`Stime` present; nothing after this
point does. They're dropped from `train_df`/`test_df` now — they still live on
permanently in `side_table` (and its `side_train`/`side_test` slices in step 8), linked
back by `flow_id`.

In [18]:
train_df = train_df.drop(columns=[c for c in ID_TIME_COLS if c in train_df.columns])
test_df = test_df.drop(columns=[c for c in ID_TIME_COLS if c in test_df.columns])

print("Train columns now:", train_df.columns.tolist())

Train columns now: ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'service', 'Sload', 'Dload', 'Spkts', 'swin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_src_ltm', 'ct_dst_sport_ltm', 'attack_cat', 'Label', 'flow_id']


### 7h. Handle class imbalance — train only
Test balance is never touched — it has to reflect the real-world class distribution, or
every metric computed on it (recall, F1, confusion matrix) stops meaning anything.

For a GNN specifically, oversampling raw feature rows (e.g. SMOTE) creates synthetic
rows that don't correspond to genuine flows, and once flows become graph nodes/edges that
distorts the graph's actual topology. Class weights (applied inside the loss function)
get the same "pay more attention to rare classes" effect without touching the graph at
all — that's the default here. It also suits this dataset better: with `Worms` at 174
raw rows total, there's very little real signal for synthetic oversampling to interpolate
from in the first place.

In [19]:
def compute_class_weights(train_df, label_col):
    classes = np.sort(train_df[label_col].unique())
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df[label_col])
    return dict(zip(classes, weights))

binary_class_weights = compute_class_weights(train_df, 'Label')
print("Binary (Label) class weights:", binary_class_weights)

# Multi-class weights too, in case the GNN target ends up being the 10-way attack_cat
# rather than the binary Label.
attack_cat_classes = np.array(sorted(train_df['attack_cat'].unique()))
attack_cat_weights_arr = compute_class_weight(
    class_weight='balanced', classes=attack_cat_classes, y=train_df['attack_cat']
)
attack_cat_class_weights = dict(zip(attack_cat_classes, attack_cat_weights_arr))

print("\nMulti-class (attack_cat) class weights:")
for k, v in attack_cat_class_weights.items():
    print(f"  {k}: {v:.3f}")

Binary (Label) class weights: {np.int64(0): np.float64(0.5209416038176136), np.int64(1): np.float64(12.437958629020082)}

Multi-class (attack_cat) class weights:
  Analysis: 110.796
  Backdoors: 127.419
  DoS: 42.715
  Exploits: 8.876
  Fuzzers: 11.060
  Generic: 10.248
  Normal: 0.104
  Reconnaissance: 18.456
  Shellcode: 161.999
  Worms: 1432.637


## 8. Save outputs
`train_processed` / `test_processed` are the model-input frames (no IP/port/time
columns). `side_train` / `side_test` are the matching IP/port/time slices, sliced from
the master `side_table` by `flow_id` so they line up row-for-row with the processed
frames. Everything is saved as both `.pkl` (preserves dtypes exactly) and `.csv` (easy to
eyeball), plus the fitted transforms so teammates can apply identical preprocessing to
any new data without refitting on it.

In [20]:
OUT_DIR = '/content/task2_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

side_train = side_table[side_table['flow_id'].isin(train_df['flow_id'])].reset_index(drop=True)
side_test = side_table[side_table['flow_id'].isin(test_df['flow_id'])].reset_index(drop=True)

def save_outputs(name, obj, out_dir=OUT_DIR):
    pkl_path = os.path.join(out_dir, f'{name}.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(obj, f)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(os.path.join(out_dir, f'{name}.csv'), index=False)
    print("Saved:", pkl_path)

save_outputs('train_processed', train_df)
save_outputs('test_processed', test_df)
save_outputs('side_train', side_train)
save_outputs('side_test', side_test)

save_outputs('fitted_transforms', {
    'fill_values': fill_values,
    'cat_encoder': cat_encoder,
    'scaler': scaler,
    'dropped_columns': DROP_COLS,
    'binary_class_weights': binary_class_weights,
    'attack_cat_class_weights': attack_cat_class_weights,
})

print("\nAll outputs saved to:", OUT_DIR)

Saved: /content/task2_outputs/train_processed.pkl
Saved: /content/task2_outputs/test_processed.pkl
Saved: /content/task2_outputs/side_train.pkl
Saved: /content/task2_outputs/side_test.pkl
Saved: /content/task2_outputs/fitted_transforms.pkl

All outputs saved to: /content/task2_outputs


## 9. Final summary

In [21]:
print("=" * 60)
print("TASK 2 PREPROCESSING SUMMARY")
print("=" * 60)

print(f"\nRaw rows loaded (4 files, pre-dedup): {rows_before_dedup:,}")
print(f"Rows after global dedup:              {rows_after_dedup:,}  "
      f"({rows_before_dedup - rows_after_dedup:,} removed)")

print(f"\nMissing values before fill:")
print(train_missing_before if len(train_missing_before) else "  (none)")
print(f"\nMissing values after fill: 0 (all filled from train-only values)")

print(f"\nSplit strategy: {SPLIT_STRATEGY}")
print(f"Train shape (final): {train_df.shape}")
print(f"Test shape (final):  {test_df.shape}")

print(f"\nColumns dropped (near-constant / high-corr): {DROP_COLS}")
print(f"ID/time columns removed from model input, kept in side table: {ID_TIME_COLS}")

print(f"\nTrain Label balance:\n{train_df['Label'].value_counts(normalize=True).round(4)}")
print(f"\nTest Label balance:\n{test_df['Label'].value_counts(normalize=True).round(4)}")
print(f"\nTrain attack_cat counts:\n{train_df['attack_cat'].value_counts()}")
print(f"\nTest attack_cat counts:\n{test_df['attack_cat'].value_counts()}")

TASK 2 PREPROCESSING SUMMARY

Raw rows loaded (4 files, pre-dedup): 2,540,047
Rows after global dedup:              2,059,415  (480,632 removed)

Missing values before fill:
is_ftp_login        37.084196
ct_flw_http_mthd    34.393262
dtype: float64

Missing values after fill: 0 (all filled from train-only values)

Split strategy: time
Train shape (final): (1647532, 32)
Test shape (final):  (411883, 32)

Columns dropped (near-constant / high-corr): ['is_sm_ips_ports', 'dwin', 'Dpkts', 'dloss', 'sloss', 'ct_src_dport_ltm', 'ct_dst_ltm', 'ct_srv_dst', 'ct_dst_src_ltm', 'synack', 'ackdat', 'ct_state_ttl']
ID/time columns removed from model input, kept in side table: ['srcip', 'sport', 'dstip', 'dsport', 'Stime', 'Ltime']

Train Label balance:
Label
0    0.9598
1    0.0402
Name: proportion, dtype: float64

Test Label balance:
Label
0    0.9189
1    0.0811
Name: proportion, dtype: float64

Train attack_cat counts:
attack_cat
Normal            1581302
Exploits            18561
Generic        